# AdaFace Noise Study - Colab Launcher

This notebook is the **EXECUTION LAUNCHER** for the AdaFace Noise Study.
It orchestrates Google Colab GPU training, pulls the exact source code from GitHub, syncs heavy datasets from Google Drive, and pushes results back.

## 1. System Verification & Mount

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Obtain Source Code (Git)

In [ ]:
import os
import sys
SOURCE_GIT_REF = "colab-0pct-v2"
REPO_URL = "https://github.com/Raja2027/adaFace-noise-study.git"

# Clean any prior checkout to ensure fresh state
if os.path.exists('/content/adaFace-noise-study'):
    !rm -rf /content/adaFace-noise-study

!git clone {REPO_URL} /content/adaFace-noise-study
os.chdir('/content/adaFace-noise-study')
!git fetch origin --tags -f

# Capture LAUNCHER commit before checking out source
launcher_commit_raw = !git rev-parse HEAD
LAUNCHER_COMMIT = launcher_commit_raw[0].strip()
print(f"Launcher commit: {LAUNCHER_COMMIT}")

# Checkout the strictly pinned source tag
!git checkout {SOURCE_GIT_REF}
!git submodule update --init --recursive

# Verify exact hashes
source_commit = !git rev-parse HEAD
source_commit = source_commit[0].strip()
print(f"Source commit: {source_commit}")
assert source_commit == "a6f2a4623f852ee00c5f3fcc31da6812f02c063c", f"Source mismatch: {source_commit}"

adaface_commit = !git -C third_party/AdaFace rev-parse HEAD
adaface_commit = adaface_commit[0].strip()
print(f"AdaFace commit: {adaface_commit}")
assert adaface_commit == "c60eaa786a42c03444f3df7096dbaf9d57ae010d", f"AdaFace mismatch: {adaface_commit}"

print("Hash verification passed.")

## 3. Data Sync & Environment Setup

In [ ]:
!python colab/setup_colab.py

## 4. FINAL PREFLIGHT VERIFICATION
**DO NOT START TRAINING UNLESS THIS SUCCEEDS.**

In [ ]:
!python colab/preflight_colab.py

## 5. Execute Training (0% Baseline)
Starts the FULL 0% NOISE BASELINE experiment natively syncing to Drive.

In [ ]:
# Patch the run manifest dynamically to record BOTH launcher and source commits
with open('colab/train_colab.py', 'r') as f:
    content = f.read()
content = content.replace(
    "'git_commit': git_hash,",
    f"'LAUNCHER_GIT_COMMIT': '{LAUNCHER_COMMIT}',\n        'SOURCE_GIT_COMMIT': git_hash,"
)
with open('colab/train_colab.py', 'w') as f:
    f.write(content)

!python colab/train_colab.py \
    --config configs/noise_0.yaml \
    --sync_dir /content/drive/MyDrive/adaFace-noise-study/checkpoints/noise_0/seed_42